# 🧠 การปรับจูนละเอียดโมเดล (Fine-Tuning Models): กลยุทธ์การแช่แข็งและอัตราการเรียนรู้ (Learning Rates)

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **Fine-Tuning Models**! ในสมุดบันทึกนี้ เราจะ:
1. แยกความแตกต่างระหว่างการเรียนรู้แบบถ่ายโอน (Transfer Learning) และการปรับจูนละเอียด (Fine-Tuning)
2. สรุปกระบวนการทำงาน: การสลับหัวลัพธ์ (head swapping), การเลือกแช่แข็งเลเยอร์เฉพาะจุด (selective layer freezing) และการใช้อัตราการเรียนรู้ระดับต่ำ (low learning rates)
3. โหลดโมเดล `ResNet18` และพัฒนา **ฟังก์ชันช่วยแช่แข็งเลเยอร์ตามชื่อ (name-based layer freezing utility)** ที่มีความยืดหยุ่นใน PyTorch
4. เปรียบเทียบ 3 กลยุทธ์ในการปรับจูนละเอียด:
   - **การปรับจูนละเอียดทั้งหมด (Full Fine-Tuning):** เปิดให้ฝึกฝนได้ทุกเลเยอร์
   - **การปรับจูนละเอียดเฉพาะส่วนหัว (Head-Only Fine-Tuning):** แช่แข็งทุกเลเยอร์ที่เป็นโครงสร้างหลัก (backbone)
   - **การปรับจูนละเอียดบางส่วน (Partial Fine-Tuning):** แช่แข็งเลเยอร์ส่วนแรกๆ แต่ปล่อยให้เลเยอร์ที่อยู่ลึกและส่วนหัวสามารถฝึกฝนได้
5. เปรียบเทียบจำนวนพารามิเตอร์ที่สามารถฝึกฝนได้และภาระการคำนวณของแต่ละกลยุทธ์
6. เชื่อมโยงกลยุทธ์เหล่านี้เข้ากับอาร์กิวเมนต์การตั้งค่า `freeze` และ `lr0` ของ YOLO

เรามาเริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันเลยครับ

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# Set seed for reproducibility
torch.manual_seed(42)

## 1. การสร้างฟังก์ชันช่วยแช่แข็งเลเยอร์

เราจะพัฒนาฟังก์ชันช่วยแช่แข็งพารามิเตอร์ในโมเดล PyTorch ตามส่วนของชื่อเลเยอร์ที่ตรงกับข้อกำหนด

In [ ]:
def freeze_layers_by_name(model, match_names):
    """
    Freeze parameters if their layer name contains any of the strings in match_names.
    """
    for name, param in model.named_parameters():
        if any(match_str in name for match_str in match_names):
            param.requires_grad = False
        else:
            param.requires_grad = True

def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

## 2. การเปรียบเทียบกลยุทธ์การปรับจูนละเอียด

เรามาประเมินพารามิเตอร์ที่สามารถฝึกฝนได้สำหรับสามกลยุทธ์ยอดนิยมดังนี้ครับ:

### กลยุทธ์ A: การปรับจูนละเอียดทั้งหมด (ฝึกฝนทุกเลเยอร์)
เราจะไม่แช่แข็งเลเยอร์ใดๆ เลย แต่จะฝึกฝนด้วยอัตราการเรียนรู้ (learning rate) ที่ต่ำมากแทน

In [ ]:
model_full = models.resnet18()
model_full.fc = nn.Linear(model_full.fc.in_features, 10)

total_p, trainable_p = count_parameters(model_full)
print("--- Strategy A: Full Fine-Tuning ---")
print(f"Total Parameters    : {total_p:,}")
print(f"Trainable Parameters: {trainable_p:,}")

### กลยุทธ์ B: การปรับจูนละเอียดเฉพาะส่วนหัว (แช่แข็งตัวสกัดคุณลักษณะ)
We freeze everything except the final classification layer (`fc`).

In [ ]:
model_head = models.resnet18()
model_head.fc = nn.Linear(model_head.fc.in_features, 10)

freeze_layers_by_name(model_head, match_names=['conv', 'bn', 'layer'])

total_p, trainable_p = count_parameters(model_head)
print("--- Strategy B: Head-Only Fine-Tuning ---")
print(f"Total Parameters    : {total_p:,}")
print(f"Trainable Parameters: {trainable_p:,}")

### กลยุทธ์ C: การปรับจูนละเอียดบางส่วน (แช่แข็งเลเยอร์แรก ฝึกฝนเลเยอร์ลึก)
Early layers (conv1, bn1, layer1, layer2) detect generic features and are frozen. Deep layers (layer3, layer4) and the head are kept trainable to adapt to custom objects.

In [ ]:
model_partial = models.resnet18()
model_partial.fc = nn.Linear(model_partial.fc.in_features, 10)

freeze_layers_by_name(model_partial, match_names=['conv1', 'bn1', 'layer1', 'layer2'])

total_p, trainable_p = count_parameters(model_partial)
print("--- Strategy C: Partial Fine-Tuning ---")
print(f"Total Parameters    : {total_p:,}")
print(f"Trainable Parameters: {trainable_p:,}")

สังเกตความแตกต่างกันครับ:
-   **กลยุทธ์ A (ทั้งหมด):** มีพารามิเตอร์ที่สามารถฝึกฝนได้ถึง $11.2$ ล้านตัว มีความยืดหยุ่นสูง แต่มีความเสี่ยงสูงที่จะเกิด **การลืมเลือนอย่างกะทันหัน (Catastrophic Forgetting)** หากอัตราการเรียนรู้กว้างเกินไป
-   **กลยุทธ์ B (เฉพาะส่วนหัว):** มีพารามิเตอร์ที่ฝึกฝนได้เพียง $5,130$ ตัว ฝึกฝนได้รวดเร็วมาก โอกาสเกิด overfitting ต่ำ แต่อาจเกิดปัญหา underfit ได้หากรูปภาพเป้าหมายแตกต่างจากชุดข้อมูล COCO อย่างสิ้นเชิง
-   **กลยุทธ์ C (บางส่วน):** มีพารามิเตอร์ที่ฝึกฝนได้ $9.4$ ล้านตัว ช่วยรักษาระบบการสกัดคุณลักษณะขอบระดับล่างไว้ ในขณะที่ปรับแต่งเลเยอร์ลึกเพื่อเรียนรู้รูปทรงวัตถุเฉพาะตัวใหม่ๆ นี่เป็นกลยุทธ์ที่เหมาะสมที่สุดสำหรับชุดข้อมูลขนาดปานกลาง

## 💡 การเชื่อมโยงกับ YOLO และการเรียนรู้เชิงลึก (Deep Learning)
*   **อาร์กิวเมนต์ `freeze`:** ใน YOLO การตั้งค่า `freeze=10` จะเป็นการแช่แข็งเลเยอร์ 10 เลเยอร์แรกของโมเดล
*   สถาปัตยกรรมของ YOLO ประกอบด้วยโครงสร้างหลักหรือ backbone (สกัดคุณลักษณะระดับต้น) และส่วนคอ/ส่วนหัว (ประสานคุณลักษณะหลายระดับและการจัดกลุ่มกรอบขอบเขต) การแช่แข็งโครงสร้างหลักจะช่วยรับประกันว่าพารามิเตอร์เชิงพื้นที่ที่ฝึกมาดีแล้วจะไม่ถูกทำลาย ซึ่งสำคัญมากเมื่อเรียนรู้จากชุดข้อมูลเป้าหมายขนาดเล็ก
*   **อัตราการเรียนรู้เริ่มต้นที่ต่ำกว่า (`lr0`):** การฝึกโมเดลปกติจากศูนย์จะใช้ `lr0=0.01` แต่การปรับจูนละเอียดมักจะลดค่านี่ลงเหลือ `lr0=0.001` หรือ `lr0=0.0001` เพื่อหลีกเลี่ยงการทำลายตัวกรองที่ผ่านการฝึกฝนมาแล้วเป็นอย่างดี